Building a simple rag for querying into pdfs
Techniques Used:
- Hybrid Retriever
- LCEL Rag Chain

In [45]:
#libraries:
from langchain_community.document_loaders import PyPDFLoader,DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain.chat_models import init_chat_model
from typing import List
from langchain_classic.retrievers import BM25Retriever,EnsembleRetriever
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

Reading PDF Files from the directory

In [ ]:
dir_loader = DirectoryLoader(
    path="data/",
    glob="*.pdf",
    loader_cls=PyPDFLoader
)
raw_pdfs = dir_loader.load()
print(len(raw_pdfs))
print(type(raw_pdfs))
print(type(raw_pdfs[0]))

50
<class 'list'>
<class 'langchain_core.documents.base.Document'>


Custom PDF Content Processor to clean and process the raw pdfs and create chunks w rich metadata:

In [32]:
class PDFProcessor:
    def __init__(self,chunk_size=1000,chunk_overlap=200):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size = chunk_size,
            chunk_overlap = chunk_overlap,
            separators = [" "]
        )
    
    def process_pdf_docs(self,raw_docs: List[Document]) -> List[Document]:
        # final chunks list to returrn
        final_chunks=[]
        pages_skipped = 0
        entire_page_as_chunk = 0

        for doc in raw_docs:
            cleaned_text = self._clean_text(doc.page_content)

            #skipping empty page
            if len(cleaned_text.strip())<50:
                pages_skipped += 1
                continue

            if len(cleaned_text)<=self.chunk_size:
                temp_chunk = Document(
                    page_content=cleaned_text,
                    metadata={
                        "source": doc.metadata.get("source"),
                        "page_num": doc.metadata.get("page_label"),
                        "total_pages": doc.metadata.get("total_pages"),
                        "creation_date": doc.metadata.get("creationdate"),
                        "chunk_method": "pdfprocessor",
                        "char_count": len(cleaned_text)
                    }
                )
                final_chunks.append(temp_chunk)
                entire_page_as_chunk += 1
                continue
            
            chunks = self.splitter.create_documents(
                texts=[cleaned_text],
                metadatas=[
                    {
                        "source": doc.metadata.get("source"),
                        "page_num": doc.metadata.get("page_label"),
                        "total_pages": doc.metadata.get("total_pages"),
                        "creation_date": doc.metadata.get("creationdate"),
                        "chunk_method": "pdfprocessor",
                        "char_count": len(cleaned_text)
                    }
                ]
            )

            for chunk in chunks:
                chunk.metadata["char_count"] = len(chunk.page_content)

            final_chunks.extend(chunks)
        print(f"Pages Skipped: {pages_skipped}")
        print(f"No of entire page as chunks: {entire_page_as_chunk}")
        return final_chunks

        
    def _clean_text(self,text: str) -> str:
        # removing white spaces
        text = " ".join(text.split())
        return text     

Chunking using the PDFProcessor

In [33]:
processor = PDFProcessor()
chunks = processor.process_pdf_docs(raw_pdfs)
print(f"No of chunks built: {len(chunks)}")
print("Sample Chunk:")
print(f"Page Content: {chunks[0].page_content[:200]}")
print("Metadata")
print(chunks[0].metadata)

Pages Skipped: 0
No of entire page as chunks: 46
No of chunks built: 54
Sample Chunk:
Page Content: Unit 2 – RPC, Distributed Objects & Communication (Simplified Detailed Notes) These notes explain every topic and subtopic from Unit 2 in a detailed yet easy-to- understand way. Difficult concepts are
Metadata
{'source': 'data\\U_2.pdf', 'page_num': '1', 'total_pages': 18, 'creation_date': '2026-05-09T21:52:08+05:30', 'chunk_method': 'pdfprocessor', 'char_count': 999}


Initializing HuggingFace Embedding Model - all-MiniLM-L6-v2

In [25]:
embedding_model = HuggingFaceEmbeddings(
    model_name = "all-MiniLM-L6-v2"
)
embedding_model

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

FAISS Vectorstore and Retriever Setup

In [34]:
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)
dense_retriever = vectorstore.as_retriever(
    search_kwargs={"k":3}
)
dense_retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001C8673441A0>, search_kwargs={'k': 3})

Sparse Retriever BM25

In [35]:
sparse_retriever = BM25Retriever.from_documents(chunks)
sparse_retriever.k = 3
sparse_retriever

BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x000001C867344AD0>, k=3)

Hybrid Retriever - combining both using Ensembke Retriever

In [36]:
hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever,sparse_retriever],
    weights=[0.7,0.3]
)
hybrid_retriever

EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001C8673441A0>, search_kwargs={'k': 3}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x000001C867344AD0>, k=3)], weights=[0.7, 0.3])

Testing the Retriever

In [37]:
query = "List the problems in RPC and what is RPC?"
result = hybrid_retriever.invoke(query)
for i,doc in enumerate(result):
    print(f"Document{i+1}:\n{doc.page_content}")

Document1:
Unit 2 – RPC, Distributed Objects & Communication (Simplified Detailed Notes) These notes explain every topic and subtopic from Unit 2 in a detailed yet easy-to- understand way. Difficult concepts are simplified using step-by-step explanations, real-life examples, and comparisons so they are easier to learn for exams. Introduction to Remote Procedure Call (RPC) Remote Procedure Call (RPC) allows a program on one computer to execute a function on another computer as if the function were local. Main Idea: The programmer writes code normally, while RPC hides networking complexity. Example: Suppose a food delivery app running on your phone needs restaurant data from another server. Instead of manually handling: - sockets - packets - network protocols The app simply calls: getRestaurantDetails() The actual function runs on another machine, but it feels like a normal function call. RPC is important because: - It simplifies distributed programming - Hides communication details - Ma

Initializing the LLM

In [38]:
llm = init_chat_model("groq:llama-3.1-8b-instant")
llm

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001C8677906E0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001C867791D30>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

Prompt for the LLM

In [41]:
prompt = PromptTemplate.from_template("""
Use the following context given below and answer the following question.
Context:
{context}
Question: {question}
IMPORTANT INSTRUCTIONS:
    - Answer in short detailed format
    - Dont add extra lines explanaing yourself just return the answer
""")

Format Function for retrieved docs

In [44]:
def format_docs(docs):
    content = []
    for doc in docs:
        content.append(doc.page_content)
    return "\n".join(content)

LCEL Rag Chain

In [46]:
rag_chain = (
    {
        "context": hybrid_retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)
rag_chain

{
  context: EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001C8673441A0>, search_kwargs={'k': 3}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x000001C867344AD0>, k=3)], weights=[0.7, 0.3])
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='\nUse the following context given below and answer the following question.\nContext:\n{context}\nQuestion: {question}\nIMPORTANT INSTRUCTIONS:\n    - Answer in short detailed format\n    - Dont add extra lines explanaing yourself just return the answer\n')
| ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False,

Testing Rag Chain

In [47]:
query = "List the problems in RPC and what is RPC?"
answer = rag_chain.invoke(query)
print(f"Question: {query}")
print("Answer:\n",answer)

Question: List the problems in RPC and what is RPC?
Answer:
 RPC (Remote Procedure Call):
- Allows a program to execute a function on another computer as if the function were local.
- Simplifies distributed programming.
- Hides communication details.
- Makes remote communication easier.

Problems in RPC:
1. Different Memory Spaces
- Client and server run on different machines, making it difficult to share pointers directly.
2. Different Data Formats
- Different computers may store data differently (e.g., big-endian, little-endian).
3. Network Delays
- Remote calls are much slower than local calls.
4. Failures
- Network failure, server crash, client crash.
5. Security Issues
- Messages may be intercepted or modified.
6. Naming Problem
- Client must locate which server and procedure to call.


In [48]:
query = "List the Dependability Requirements and what is Dependability?"
answer = rag_chain.invoke(query)
print(f"Question: {query}")
print("Answer:\n",answer)

Question: List the Dependability Requirements and what is Dependability?
Answer:
 **Dependability:**
"Dependability means: How trustworthy and reliable the system is."

**Dependability Requirements:**

1. **Availability:** The system is ready to use whenever users need it.
2. **Reliability:** The system works continuously without failure for a long time.
3. **Safety:** Failures should not cause disasters.
4. **Maintainability:** How easily can the system be repaired?
5. **Security:** Protects data integrity, confidential information, and system resources


In [49]:
query = "What is Synchronization and why Synchronization is Difficult in Distributed Systems?"
answer = rag_chain.invoke(query)
print(f"Question: {query}")
print("Answer:\n",answer)

Question: What is Synchronization and why Synchronization is Difficult in Distributed Systems?
Answer:
 **What is Synchronization?**
Synchronization means coordinating multiple processes or computers so they work together correctly. It ensures that processes do not interfere with each other, shared resources are used safely, and events occur in the correct order.

**Why Synchronization is Difficult in Distributed Systems?**
Synchronization is difficult in distributed systems because of the following reasons:
- Different machines have different clocks.
- No shared memory exists.
- Communication delays occur.
- Machines may fail independently.
- There is no single system clock, making it harder to coordinate processes.
- Network delays can cause messages to arrive later than expected, leading to incorrect decisions.
